# Lab 2: Discovery of Frequent Itemsets and Association

In [5]:
with open("../datasets/transactions.dat", "rb") as f:
    chunk = f.read(300)

print(chunk)

b'25 52 164 240 274 328 368 448 538 561 630 687 730 775 825 834 \n39 120 124 205 401 581 704 814 825 834 \n35 249 674 712 733 759 854 950 \n39 422 449 704 825 857 895 937 954 964 \n15 229 262 283 294 352 381 708 738 766 853 883 966 978 \n26 104 143 320 569 620 798 \n7 185 214 350 529 658 682 782 809 849 883'


## Setup

First install all of the required dependencies.

In [6]:
!sudo apt-get update -qq
!sudo apt-get install -y -qq --no-install-recommends openjdk-17-jre-headless
%pip install -q pyspark sympy matplotlib

debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 10.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (Can't locate Term/ReadLine.pm in @INC (you may need to install the Term::ReadLine module) (@INC entries checked: /etc/perl /usr/local/lib/x86_64-linux-gnu/perl/5.40.1 /usr/local/share/perl/5.40.1 /usr/lib/x86_64-linux-gnu/perl5/5.40 /usr/share/perl5 /usr/lib/x86_64-linux-gnu/perl-base /usr/lib/x86_64-linux-gnu/perl/5.40 /usr/share/perl/5.40 /usr/local/lib/site_perl) at /usr/share/perl5/Debconf/FrontEnd/Readline.pm line 8, <STDIN> line 10.)
debconf: falling back to frontend: Teletype
Selecting previously unselected package libsqlite3-0:amd64.
(Reading database ... 9462 files and directories currently installed.)
Preparing to unpack .../0-libsqlite3-0_3.46.1-9ubuntu0.3_a

Next up, create the instance of Spark and configure it with basic parameters.

In [7]:
import os, sys, subprocess
os.environ["PYSPARK_PYTHON"] = sys.executable

print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr)

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("lab1")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.network.timeout", "600s")
         .config("spark.executor.heartbeatInterval", "60s")
         .getOrCreate())

sc = spark.sparkContext

print(spark.version)

openjdk version "17.0.21.0.3" 2026-10-24
OpenJDK Runtime Environment (build 17.0.21.0.3+1-1-26.04.1-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.21.0.3+1-1-26.04.1-Ubuntu, mixed mode, sharing)



Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/15 14:35:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.3.0


## Finding frequent itemsets

In [9]:
baskets = (sc.textFile("../datasets/transactions.dat")
            .map(lambda line: tuple(sorted(set(map(int, line.split())))))
            .filter(len)
            .cache())

print(baskets.count())

[Stage 0:>                                                          (0 + 2) / 2]

100000


In [13]:
from itertools import combinations
from operator import add

class Apriori:
    def __init__(self, support: float):
        self.s = support

    def find_frequent_itemsets(self, baskets_rdd) -> dict[int, dict[tuple, int]]:
        sc = baskets_rdd.context
        min_count = self.s * baskets_rdd.count()
        result = {}

        singles = (baskets_rdd.flatMap(lambda b: b)
                              .map(lambda x: ((x,), 1))
                              .reduceByKey(add)
                              .filter(lambda kv: kv[1] >= min_count)
                              .collectAsMap())
        result[1] = singles

        frequent_items = sorted(t[0] for t in singles)

        k = 2

        while result[k - 1]:
            candidates = self._generate_candidates(set(result[k - 1]), frequent_items, k)

            if not candidates:
                break

            bc = sc.broadcast(candidates)

            counts = (baskets_rdd
                      .flatMap(lambda b: [(c, 1) for c in combinations(b, k) if c in bc.value])
                      .reduceByKey(add)
                      .filter(lambda kv: kv[1] >= min_count)
                      .collectAsMap())

            bc.unpersist()

            if not counts:
                break

            result[k] = counts
            k += 1

        return result

    def _generate_candidates(self, prev_frequent: set[tuple], frequent_items: list[int], k: int) -> set[tuple]:
        candidates = set()

        for itemset in prev_frequent:
            last = itemset[-1]

            for x in frequent_items:
                if x <= last:
                    continue

                candidate = itemset + (x,)

                if all(sub in prev_frequent for sub in combinations(candidate, k - 1)):
                    candidates.add(candidate)

        return candidates

In [14]:
apriori = Apriori(support=0.01)

levels = apriori.find_frequent_itemsets(baskets)
print(levels[3])

[Stage 20:>                                                         (0 + 2) / 2]

{(39, 704, 825): 1035}


## Generating association rules

## Evaluation

## Conclusions